# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`

This notebook demonstrates how to load and explore the [FAIR<sup>2</sup>](https://sen.science/doi/10.71728/senscience.vq4a-28xa) dataset, which details the analysis of human protein abundance, modifications, and sequences, using the [`mlcroissant`](https://github.com/mlcommons/croissant) library. You will learn to load Croissant metadata, inspect available record sets, extract data by referencing `@id`s, and perform initial exploratory analysis in pandas.

### Dataset Source
The dataset source is provided via a Croissant schema URL:
- https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load the dataset schema and metadata using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
# Convert to plain Python dict for pretty printing
metadata = dataset.metadata.to_json()

print(f"Dataset Name: {metadata.get('name')}")
print(f"Description: {metadata.get('description')}")

## 2. Data Overview

List available record sets, their `@id`s, and the fields contained within each. All further data extraction uses these `@id` references.

In [ ]:
from pprint import pprint

# List record sets in this dataset. Each has its own `@id`.
record_sets = dataset.metadata.record_sets
if record_sets:
    for rs in record_sets:
        print(f"RecordSet name: '{rs.name}'  |  @id: '{rs.id}'")
        # List all field @id and description in each record set
        print("  Fields:")
        for field in rs.fields:
            print(f"    name: '{field.name}', @id: '{field.id}', dataType: {field.data_type}")
        print()
else:
    print("No record sets found in metadata.")

## 3. Data Extraction

For each record set found in the previous step, load records into pandas DataFrames using its `@id`. Below we demonstrate this for all available record sets.

In [ ]:
# Gather all record set @id's
record_set_ids = [rs.id for rs in dataset.metadata.record_sets]
dataframes = {}

for record_set_id in record_set_ids:
    # Load all records for the set
    try:
        records = list(dataset.records(record_set=record_set_id))
        if records:
            df = pd.DataFrame(records)
            dataframes[record_set_id] = df
            print(f"Loaded {len(df)} records from RecordSet @id '{record_set_id}'.")
            print(f"Columns: {df.columns.tolist()}")
            display(df.head(2))
        else:
            print(f"No records for RecordSet @id '{record_set_id}'.")
    except Exception as e:
        print(f"Error loading RecordSet '{record_set_id}': {e}")

# For demonstration, pick the first populated record set for downstream analysis
if dataframes:
    main_record_set_id = next(iter(dataframes))
    print(f"Defaulting to first non-empty RecordSet: {main_record_set_id}")
else:
    main_record_set_id = None
    print("No record sets were extractable.")

## 4. Exploratory Data Analysis (EDA)
Perform basic data analysis: filtering rows, normalizing a numeric field, and grouping. For this, all references are by `@id`, so please refer to the record set and field lists above. Update the `numeric_field_id` and `group_field_id` with valid choices from your record set.

In [ ]:
if main_record_set_id is not None:
    df = dataframes[main_record_set_id]
    # Show columns with their @id for reference
    print("Available columns:")
    pprint(df.columns.tolist())

    # Example: pick the first found numeric (float or int) column
    numeric_field_id = None
    # Map field @id to type
    fields_by_id = {f.id: f for f in [f for rs in dataset.metadata.record_sets for f in rs.fields]}
    for col in df.columns:
        field = fields_by_id.get(col)
        if field is not None:
            if field.data_type in ("Float", "Integer", "Number"):
                numeric_field_id = col
                break

    if numeric_field_id is not None:
        print(f"Using '{numeric_field_id}' as numeric field for EDA.")
        # Choose a threshold for filtering
        threshold = df[numeric_field_id].mean() if pd.api.types.is_numeric_dtype(df[numeric_field_id]) else 0
        print(f"Filtering records where {numeric_field_id} > {threshold:.2f}")
        filtered_df = df[df[numeric_field_id] > threshold].copy()
        print(f"Filtered {len(filtered_df)} records.")
        display(filtered_df.head())

        # Normalization
        filtered_df[f"{numeric_field_id}_normalized"] = (
            (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        )
        print(f"Normalized '{numeric_field_id}':")
        display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Try grouping by a categorical field
        group_field_id = None
        # Prefer string field not equal to the numeric
        for col in df.columns:
            field = fields_by_id.get(col)
            # Heuristically pick first text/categorical
            if field is not None and field.data_type in ("Text", "String") and col != numeric_field_id:
                group_field_id = col
                break
        if group_field_id and group_field_id in filtered_df.columns:
            print(f"Grouping by '{group_field_id}':")
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().to_frame()
            display(grouped_df.head())
        else:
            print("No suitable categorical field for grouping found.")
    else:
        print("No numeric fields found in this record set.")
else:
    print("No data available for EDA.")

## 5. Visualization
Visualize the distribution of the selected numeric field and its relationship to the chosen categorical field, using only field `@id`s.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if main_record_set_id is not None and numeric_field_id is not None:
    plt.figure(figsize=(8, 5))
    sns.histplot(data=df, x=numeric_field_id, bins=30, kde=True)
    plt.title(f"Distribution of '{numeric_field_id}'")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()

    if group_field_id is not None and group_field_id in df.columns:
        plt.figure(figsize=(10, 5))
        # Show boxplot numeric by group
        sns.boxplot(data=df, x=group_field_id, y=numeric_field_id)
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xticks(rotation=45, ha='right')
        plt.show()
else:
    print("No data available for visualization.")

## 6. Conclusion

- This notebook demonstrated access to a fully FAIR-compliant dataset using `mlcroissant`.
- Record sets, fields, and all data are referenced by their Croissant `@id` for robust, schema-consistent analysis.
- You can extend this notebook for machine learning, further visualization, or domain-specific statistics by selecting relevant `@id`s for your needs.